# Sample profile and climate-stress perceptions

Produces the descriptive results in Table 4.9 and the Pearson chi-square tests in Table 4.10.

Personal file paths, stored outputs, exploratory analyses, and superseded cells have been removed.

In [ ]:
from pathlib import Path
import warnings

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name in {"notebooks", "needs_verification"}:
    PROJECT_DIR = PROJECT_DIR.parent

DATA_FILE = PROJECT_DIR / "data" / "questionnaire.xlsx"
OUTPUTS_DIR = PROJECT_DIR / "outputs"

if not DATA_FILE.exists():
    raise FileNotFoundError(
        f"Input file not found: {DATA_FILE}\n"
        "Place the anonymised questionnaire file in data/questionnaire.xlsx "
        "or update DATA_FILE."
    )

OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency
from IPython.display import display

OUT_DIR = OUTPUTS_DIR / "01_sample_profile_chi_square"
OUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_excel(DATA_FILE)
df.columns = df.columns.str.strip()

COL = {
    "gender": "What is your gender?",
    "age": "What is your age?",
    "education": "What is your educational level?",
    "farm_size": "What is the size of your farm (in acres)?",
    "drought": "Have you faced challenges in planting due to drought?",
    "yield_decline": "Have you observed a decrease in agricultural yield?",
    "irrigation": "Have you used water for irrigation during the past five years?",
    "soil_knowledge": "Do you have knowledge of the soil type in your agriculture land?",
    "extreme_damage": "Have you experienced any damages caused by extreme events?",
    "infiltration": "Do you know the concepts of soil infiltration and soil conductivity?",
    "fertilizer_risk": "Are you aware of any environmental risks associated with the excessive use or misuse of fertilizers?",
    "pesticide_risk": "Are you aware of any environmental and health risks associated with the use of pesticides?",
    "water_reduction": "Have you noticed a reduction in the availability of water from your preferred irrigation source?",
}

missing = [name for name in COL.values() if name not in df.columns]
if missing:
    raise KeyError(f"Missing questionnaire columns:\n{missing}")

def clean_text(series):
    return series.astype("string").str.strip()

def parse_farm_size_acres(series):
    return (
        series.astype("string")
        .str.replace(",", "", regex=False)
        .str.extract(r"(\d+(?:\.\d+)?)", expand=False)
        .pipe(pd.to_numeric, errors="coerce")
    )

def yes_indicator(series):
    """Return 1 for responses beginning with Yes, 0 for responses beginning with No."""
    values = clean_text(series).str.lower()
    out = pd.Series(np.nan, index=series.index, dtype=float)
    out.loc[values.str.startswith("yes", na=False)] = 1
    out.loc[values.str.startswith("no", na=False)] = 0
    return out

df["farm_size_acres"] = parse_farm_size_acres(df[COL["farm_size"]])
df["farm_size_ha"] = df["farm_size_acres"] * 0.40468564224

# qcut is applied only to valid farm-size observations, as in the retained code.
valid_size = df["farm_size_ha"].dropna()
df.loc[valid_size.index, "farm_size_quartile"] = pd.qcut(
    valid_size,
    q=4,
    labels=["Q1 (smallest)", "Q2", "Q3", "Q4 (largest)"],
)


In [ ]:
# Table 4.9: sample profile and climate-stress indicators
n_total = len(df)

def frequency_rows(series, variable, category_order=None):
    cleaned = clean_text(series)
    counts = cleaned.value_counts(dropna=False)

    if category_order is None:
        category_order = list(counts.index)

    rows = []
    for category in category_order:
        if category == "Missing":
            n = int(cleaned.isna().sum())
        else:
            n = int((cleaned == category).sum())
        rows.append({
            "Variable": variable,
            "Category": category,
            "N": n,
            "Percent": round(n / n_total * 100, 1),
        })
    return rows

table_49_rows = []
table_49_rows += frequency_rows(
    df[COL["gender"]], "Gender", ["Male", "Female"]
)
table_49_rows += frequency_rows(
    df[COL["age"]], "Age group", ["18-35", "35-50", ">50", "<18"]
)

education = clean_text(df[COL["education"]])
education_clean = pd.Series("Missing", index=df.index, dtype="string")
education_clean.loc[education.str.contains("primary", case=False, na=False)] = "Primary"
education_clean.loc[education.str.contains("secondary", case=False, na=False)] = "Secondary"
education_clean.loc[
    education.str.contains("tertiary|universit|college|technical", case=False, na=False)
] = "Tertiary"

table_49_rows += frequency_rows(
    education_clean, "Education", ["Primary", "Secondary", "Tertiary", "Missing"]
)

quartile_stats = (
    df.dropna(subset=["farm_size_quartile", "farm_size_ha"])
      .groupby("farm_size_quartile", observed=True)["farm_size_ha"]
      .agg(N="count", Mean_ha="mean", Median_ha="median")
      .reset_index()
)

for _, row in quartile_stats.iterrows():
    table_49_rows.append({
        "Variable": "Farm-size quartile",
        "Category": str(row["farm_size_quartile"]),
        "N": int(row["N"]),
        "Percent": round(row["N"] / n_total * 100, 1),
        "Mean_ha": round(row["Mean_ha"], 2),
        "Median_ha": round(row["Median_ha"], 2),
    })

for key, label in [
    ("drought", "Drought impacts"),
    ("yield_decline", "Yield decline"),
    ("irrigation", "Irrigation use"),
    ("soil_knowledge", "Soil-type knowledge"),
]:
    values = clean_text(df[COL[key]])
    for category in values.value_counts(dropna=False).index:
        category_label = "Missing" if pd.isna(category) else str(category)
        n = int(values.isna().sum()) if pd.isna(category) else int((values == category).sum())
        table_49_rows.append({
            "Variable": label,
            "Category": category_label,
            "N": n,
            "Percent": round(n / n_total * 100, 1),
        })

table_49 = pd.DataFrame(table_49_rows)
table_49.to_excel(OUT_DIR / "table_4_9_sample_profile.xlsx", index=False)
display(table_49)


In [ ]:
# Table 4.10: Pearson chi-square tests
indicator_map = {
    "Drought impacts": "drought",
    "Yield decline": "yield_decline",
    "Irrigation use": "irrigation",
    "Soil knowledge": "soil_knowledge",
    "Extreme event damage": "extreme_damage",
    "Infiltration / Conductivity": "infiltration",
    "Fertilizer risk": "fertilizer_risk",
    "Pesticide risk": "pesticide_risk",
    "Water source reduction": "water_reduction",
}

demographics = {
    "Gender": COL["gender"],
    "Age": COL["age"],
    "Education": COL["education"],
}

rows = []
for indicator, key in indicator_map.items():
    response_col = COL[key]
    yes = yes_indicator(df[response_col])
    percent_yes = round(yes.mean(skipna=True) * 100, 1)

    result = {"Indicator": indicator, "Percent_Yes": percent_yes}

    # The original retained analysis used the questionnaire response categories
    # in the contingency tables rather than collapsing them to Yes/No.
    for demo_label, demo_col in demographics.items():
        analysis_data = df[[demo_col, response_col]].dropna().copy()
        contingency = pd.crosstab(
            clean_text(analysis_data[demo_col]),
            clean_text(analysis_data[response_col]),
        )
        if contingency.shape[0] > 1 and contingency.shape[1] > 1:
            _, p_value, _, _ = chi2_contingency(contingency)
        else:
            p_value = np.nan
        result[f"p_{demo_label}"] = p_value

    rows.append(result)

table_410 = pd.DataFrame(rows)
for column in ["p_Gender", "p_Age", "p_Education"]:
    table_410[column] = table_410[column].round(3)

table_410.to_excel(OUT_DIR / "table_4_10_chi_square.xlsx", index=False)
display(table_410)
